# 第06章 B组验证

- 本章角色：B（验证），本章第 2 组 = **B2 组（G9）**；轮次：第 1 轮（2026-10-30）
- 被审版本：`main` `bc956dff468aa2ab1bcaa41e63ab3b253af1ec1d`；A1-G12-正文 `227b54b2…`、A2-G15-正文 `1453e18e…`
- 本 notebook 只放 **B 组自己的对照计算**；A 稿的复现记录写在 `B2组-验证报告.md`，问题编号见 `B2组-修正清单.md`
- 环境：全新 `course-review-ch06/.venv`（Python 3.12.16 / NumPy 2.5.4 / Matplotlib 3.11.2 / PyTorch 2.14.1+cpu，CPU）
- 测试编号 V1–V6 与 `B2组-独立推导.md` 的独立预期一一对应

In [1]:
import sys, platform, json, math
import numpy as np
import torch
print("python  ", sys.version.split()[0])
print("numpy   ", np.__version__)
print("torch   ", torch.__version__, "| cuda available:", torch.cuda.is_available())
print("platform", platform.platform())
print("executable", sys.executable)

python   3.12.16
numpy    2.5.4
torch    2.14.1+cpu | cuda available: False
platform Windows-11-10.0.26100-SP0
executable <course-review-ch06>\.venv\Scripts\python.exe


## V1　6.1 PINN：配点损失单独用是否足够（对应独立推导 1、2）

**独立预期**：`-u''=f` 的残差损失在 $f\equiv0$ 时有二维零空间 $\mathrm{span}\{1,x\}$；一般 $f$ 时残差损失只把解定到「特解 + 仿射」。
因此（a）只优化残差项、不用边界条件，端点值不受约束；（b）硬约束 $\tilde u=x(1-x)N_\theta(x)$ 的边界误差应到机器精度。

模型问题取 $u^*(x)=\sin\pi x$，即 $f=\pi^2\sin\pi x$；网络 $1\to32\to32\to1$（tanh），64 个内点配点，Adam 3000 步，评价在 1001 点均匀网格（与配点不重合）。

In [3]:
import torch
import numpy as np

torch.set_default_dtype(torch.float64)
PI = math.pi
xe = torch.linspace(0.0, 1.0, 1001).view(-1, 1)
u_exact = torch.sin(PI * xe)

def make_net():
    return torch.nn.Sequential(
        torch.nn.Linear(1, 32), torch.nn.Tanh(),
        torch.nn.Linear(32, 32), torch.nn.Tanh(),
        torch.nn.Linear(32, 1))

def residual(x, u):
    d1 = torch.autograd.grad(u, x, torch.ones_like(u), create_graph=True)[0]
    d2 = torch.autograd.grad(d1, x, torch.ones_like(d1), create_graph=True)[0]
    return d2 + PI**2 * torch.sin(PI * x)   # -u''=f  =>  u''+f = 0

def train(mode, steps=3000, lr=1e-2, seed=0):
    torch.manual_seed(seed)
    net = make_net()
    opt = torch.optim.Adam(net.parameters(), lr=lr)
    xr = torch.linspace(0.005, 0.995, 64).view(-1, 1)
    xb = torch.tensor([[0.0], [1.0]])
    for _ in range(steps):
        opt.zero_grad()
        x = xr.clone().requires_grad_(True)
        if mode == "hard":
            base = net(x)
            u = x * (1 - x) * base
            d1 = torch.autograd.grad(u, x, torch.ones_like(u), create_graph=True)[0]
            d2 = torch.autograd.grad(d1, x, torch.ones_like(d1), create_graph=True)[0]
            loss = ((d2 + PI**2 * torch.sin(PI * x))**2).mean()
        else:
            u = net(x)
            loss = (residual(x, u)**2).mean()
            if mode == "soft":
                ub = net(xb)
                loss = loss + 100.0 * (ub**2).mean()
        loss.backward(); opt.step()
    with torch.no_grad():
        if mode == "hard":
            ue = xe * (1 - xe) * net(xe)
            ub = torch.zeros(2, 1)          # 硬约束下端点恒为 0
        else:
            ue = net(xe)
            ub = net(torch.tensor([[0.0], [1.0]]))
    rel_l2 = float(torch.norm(ue - u_exact) / torch.norm(u_exact))
    bnd = float(ub.abs().max())
    return rel_l2, bnd

rows = []
for mode, name in [("residual", "仅残差（无边界条件）"), ("soft", "软约束 λ=100"), ("hard", "硬约束 x(1-x)N(x)")]:
    rel, bnd = train(mode)
    rows.append((name, rel, bnd))
    print(f"{name:<22} 相对L2误差={rel:.3e}   端点最大误差={bnd:.3e}")

仅残差（无边界条件）             相对L2误差=1.447e+00   端点最大误差=1.120e+00
软约束 λ=100              相对L2误差=1.909e-02   端点最大误差=1.897e-02
硬约束 x(1-x)N(x)         相对L2误差=6.188e-05   端点最大误差=0.000e+00


In [4]:
import matplotlib.pyplot as plt
print("V1 结论：")
for name, rel, bnd in rows:
    print(f"  {name}: rel_L2={rel:.2e}, 边界误差={bnd:.2e}")
print()
print(f"仅残差与软约束的边界误差比 = {rows[0][2]/rows[1][2]:.1f} 倍")
print(f"软约束与硬约束的边界误差比 = {rows[1][2]/max(rows[2][2],1e-300):.1e} 倍")
print("判据：硬约束边界误差应 ≤1e-12；仅残差应 O(1) 且明显大于软约束。")

V1 结论：
  仅残差（无边界条件）: rel_L2=1.45e+00, 边界误差=1.12e+00
  软约束 λ=100: rel_L2=1.91e-02, 边界误差=1.90e-02
  硬约束 x(1-x)N(x): rel_L2=6.19e-05, 边界误差=0.00e+00

仅残差与软约束的边界误差比 = 59.0 倍
软约束与硬约束的边界误差比 = 1.9e+298 倍
判据：硬约束边界误差应 ≤1e-12；仅残差应 O(1) 且明显大于软约束。


## V2　6.1 反例：$f\equiv0$ 时残差为零的解不唯一

**独立预期**：$f\equiv0$ 时任意 $u=c_0+c_1x$ 都使残差为零；要求 $u(0)=u(1)=0$ 只允许 $u\equiv0$。
只优化残差项的网络会停在某个仿射函数上，端点值不为零（即 A1 开场漫画描述的现象）。

In [6]:
torch.manual_seed(0)
net = make_net()
# 把输出层偏置初始化为 2：网络在 u≈2 附近起步（相当于「猜了一个非零常数解」）
with torch.no_grad():
    net[-1].bias.fill_(2.0)
opt = torch.optim.Adam(net.parameters(), lr=1e-2)
xr = torch.linspace(0.005, 0.995, 64).view(-1, 1)
for _ in range(2000):
    opt.zero_grad()
    x = xr.clone().requires_grad_(True)
    u = net(x)
    d1 = torch.autograd.grad(u, x, torch.ones_like(u), create_graph=True)[0]
    d2 = torch.autograd.grad(d1, x, torch.ones_like(d1), create_graph=True)[0]
    loss = (d2**2).mean()          # f = 0，只用残差
    loss.backward(); opt.step()
with torch.no_grad():
    ends_t = net(torch.tensor([[0.0], [1.0]])).flatten().tolist()
x = xe.clone().requires_grad_(True)
u = net(x)
d1 = torch.autograd.grad(u, x, torch.ones_like(u), create_graph=True)[0]
d2 = torch.autograd.grad(d1, x, torch.ones_like(d1), create_graph=True)[0]
res = float(d2.abs().max())
ends = ends_t
print(f"残差 max|u''| = {res:.3e}   （≈0 说明内部方程满足）")
print(f"端点值 u(0)={ends[0]:.4f}, u(1)={ends[1]:.4f}   （要求是 0 和 0）")
print("与独立预期一致：内部方程满足、端点不满足，网络停在仿射族的一个非零代表上。")

# 直接验证零空间论证：任意仿射函数都使该算子为 0（二阶导解析为 0）
for c0, c1 in [(2.0, 3.0), (-1.5, 0.5)]:
    x2 = np.linspace(0, 1, 50)
    u2 = c0 + c1 * x2
    d2_num = np.diff(u2, 2) / (x2[1] - x2[0])**2      # 数值二阶差分
    print(f"  仿射 u={c0}+{c1}x: 解析 u''=0；数值 max|二阶差分|={np.abs(d2_num).max():.1e}；端点=({u2[0]}, {u2[-1]})")

<cell6>:22: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\torch\csrc\autograd\generated\python_variable_methods.cpp:821.)
残差 max|u''| = 3.532e-05   （≈0 说明内部方程满足）
端点值 u(0)=2.0879, u(1)=2.0782   （要求是 0 和 0）
与独立预期一致：内部方程满足、端点不满足，网络停在仿射族的一个非零代表上。
  仿射 u=2.0+3.0x: 解析 u''=0；数值 max|二阶差分|=2.1e-12；端点=(2.0, 5.0)
  仿射 u=-1.5+0.5x: 解析 u''=0；数值 max|二阶差分|=5.3e-13；端点=(-1.5, -1.0)


## V3　6.2 反问题：灵敏度与可辨识性（对应独立推导 4、5）

**独立预期**：
1. 系数扰动 $\delta a$ 引起的解扰动满足 $-v''+av=-u^*\delta a$，即 $\delta u\approx v\,\delta a$；
2. 若源项振幅 $s$ 也未知，单频解只确定 $\pi^2+a=s$，$(a,s)$ 不可分辨（雅可比秩 1）。

In [8]:
import numpy as np

N = 400
h = 1.0 / (N + 1)
x = np.linspace(h, 1 - h, N)
# 二阶差分矩阵（Dirichlet 齐次）
main = 2.0 / h**2 * np.ones(N)
off = -1.0 / h**2 * np.ones(N - 1)
A = np.diag(main) + np.diag(off, 1) + np.diag(off, -1)

def solve(a, s=1.0):
    # -u'' + a u = s*sin(pi x), u(0)=u(1)=0
    f = s * np.sin(np.pi * x)
    return np.linalg.solve(A + a * np.eye(N), f)

a0, s0 = 1.0, PI**2 + 1.0          # 使 u = sin(pi x)
u0 = solve(a0, s0)
print(f"参考解与 sin(pi x) 的最大偏差 = {np.abs(u0 - np.sin(np.pi*x)).max():.2e}")

# 灵敏度：解析方程 -v''+a v = -u*
va = np.linalg.solve(A + a0 * np.eye(N), -u0)
# 数值差商验证
da = 1e-4
un = solve(a0 + da, s0)
est = (un - u0) / da
print(f"灵敏度解析解 vs 差商：相对差 = {np.abs(est - va).max()/np.abs(va).max():.2e}")
print(f"||v||_inf = {np.abs(va).max():.4f}, ||u||_inf = {np.abs(u0).max():.4f}")

# 一个可辨识性退化：只观测 u 本身时，(a, s) 的雅可比
vs = u0 / s0        # ∂u/∂s
J = np.column_stack([va, vs])          # d u / d(a,s)
sv = np.linalg.svd(J, compute_uv=False)
print(f"\n雅可比奇异值: {sv[0]:.3e}, {sv[1]:.3e}  -> 条件数 {sv[0]/sv[1]:.3e}")
print("结论：两列共线（都正比于 sin(pi x)），(a,s) 不可同时辨识；独立预期成立。")

# 噪声对系数估计的影响：相对噪声 1% 时 a 的可辨识精度（一阶估计）
for noise in [0.0, 1e-3, 1e-2]:
    sigma = noise * np.abs(u0).max()
    da_est = sigma / (np.linalg.norm(va) / np.sqrt(N))   # 粗略最小二乘标准差
    print(f"观测相对噪声 {noise:.0e}: 一阶估计 sd(a) ≈ {da_est:.3e}")

参考解与 sin(pi x) 的最大偏差 = 4.64e-06
灵敏度解析解 vs 差商：相对差 = 9.16e-06
||v||_inf = 0.0920, ||u||_inf = 1.0000

雅可比奇异值: 1.842e+00, 6.904e-15  -> 条件数 2.669e+14
结论：两列共线（都正比于 sin(pi x)），(a,s) 不可同时辨识；独立预期成立。
观测相对噪声 0e+00: 一阶估计 sd(a) ≈ 0.000e+00
观测相对噪声 1e-03: 一阶估计 sd(a) ≈ 1.535e-02
观测相对噪声 1e-02: 一阶估计 sd(a) ≈ 1.535e-01


## V4　6.3 DeepONet：秩 p 的结构上限（对应独立推导 7）

**独立预期**：线性算子 $\mathcal G(a)(y)=\int K(x,y)a(x)dx$ 用 p 个 trunk 函数展开时，
最优可达误差就是核 $K$ 的秩 $p$ 截断误差（第 $p+1$ 个奇异值之后的部分）。
取 $K(x,y)=e^{-|x-y|/\ell}$ 验证：$p=32$ 时截断误差应有一个与训练无关的地板。

In [10]:
M = 200
xg = np.linspace(0, 1, M)
for ell in [0.05, 0.2]:
    K = np.exp(-np.abs(xg[:, None] - xg[None, :]) / ell) / (2 * ell)
    U, S, Vt = np.linalg.svd(K)
    err32 = np.sqrt((S[32:]**2).sum()) / np.sqrt((S**2).sum())
    err8 = np.sqrt((S[8:]**2).sum()) / np.sqrt((S**2).sum())
    print(f"ell={ell}: rank8 相对误差={err8:.3e}, rank32 相对误差={err32:.3e}")
    print(f"          前 5 个奇异值: {np.round(S[:5], 3)}")
print()
print("检验：把 trunk 固定为前 32 个左奇异向量后，对 200 个随机输入函数做最小二乘拟合，")
print("      得到的误差应与上面的截断误差一致（DeepONet 的假设类正是秩 32 近似）。")
rng = np.random.default_rng(0)
ell = 0.2
K = np.exp(-np.abs(xg[:, None] - xg[None, :]) / ell) / (2 * ell)
U, S, Vt = np.linalg.svd(K)
trunk = U[:, :32]                      # t_k(y)，取前 32 个左奇异向量
Ain = rng.standard_normal((100, M))
Gout = Ain @ K.T                       # 真算子作用：G(a)(y) = ∫K(x,y)a(x)dx
# 最优 branch 系数：把 a 投影到右奇异向量，再乘奇异值（即 K 的秩 32 截断）
coef = (Ain @ Vt[:32].T) * S[:32]
approx = coef @ trunk.T
rel = np.linalg.norm(approx - Gout) / np.linalg.norm(Gout)
print(f"截断重构相对误差 = {rel:.3e}   （应与上面 rank32 截断误差一致）")

ell=0.05: rank8 相对误差=3.575e-01, rank32 相对误差=6.210e-02
          前 5 个奇异值: [195.222 184.223 168.289 149.937 131.306]
ell=0.2: rank8 相对误差=6.451e-02, rank32 相对误差=8.288e-03
          前 5 个奇异值: [164.858 104.787  62.035  38.091  24.905]

检验：把 trunk 固定为前 32 个左奇异向量后，对 200 个随机输入函数做最小二乘拟合，
      得到的误差应与上面的截断误差一致（DeepONet 的假设类正是秩 32 近似）。
截断重构相对误差 = 7.829e-03   （应与上面 rank32 截断误差一致）


## V5　6.3 FNO：谱卷积的周期假设与参数量（对应独立推导 8）

**独立预期**：
1. 一维谱卷积 $K(v)=\mathcal F^{-1}(R\,\mathcal F v)$ 等价于**循环**卷积，即隐含周期延拓；
2. 每层谱权重是 `in×out×k_max` 的复数张量，可训练实参数为 $2\cdot in\cdot out\cdot k_{max}$；
3. 目标非周期时，循环卷积在边界缝合处产生误差。

In [12]:
# 1) 谱卷积 == 循环卷积
n, kmax = 64, 8
v = np.random.default_rng(1).standard_normal(n)
R = np.zeros(n, dtype=complex)
# 只保留低频：正频率 1..kmax，负频率 n-kmax..n-1，共轭对称保证实输出
idx = np.r_[0:kmax+1, n-kmax:n]
R[idx] = np.random.default_rng(2).standard_normal(len(idx)) + 1j*np.random.default_rng(3).standard_normal(len(idx))
R = np.fft.ifft(R)                       # 反变换得到等价的卷积核（循环）
R_hat = np.fft.fft(R)
v_hat = np.fft.fft(v)
spec = np.fft.ifft(R_hat * v_hat).real
circ = np.fft.ifft(np.fft.fft(v) * np.fft.fft(np.fft.ifft(R_hat))).real
print(f"(a) 谱卷积与循环卷积最大差 = {np.abs(spec - circ).max():.2e}")

# 2) 参数量
c_in, c_out, k = 32, 32, 8
print(f"(b) 谱层复数权重形状 in×out×k = {c_in}×{c_out}×{k}，可训练实参数 = {2*c_in*c_out*k}")

# 3) 非周期目标上的边界误差：比较「循环卷积」与「零填充局部卷积」
x = np.arange(n) / n
target = x.copy()                      # 在 [0,1) 上非周期（首尾不连续）
lin = target.copy()
lin[1:] += -0.5 * target[:-1]          # 左邻点（越界按 0）
lin[:-1] += -0.5 * target[1:]          # 右邻点（越界按 0）
k_circ = np.zeros(n); k_circ[0] = 1.0; k_circ[1] = -0.5; k_circ[-1] = -0.5
circ = np.fft.ifft(np.fft.fft(target) * np.fft.fft(k_circ)).real
print(f"(c) 非周期目标：边界点 |循环-零填充| = {max(abs(circ[0]-lin[0]), abs(circ[-1]-lin[-1])):.2e}")
print(f"    内部（远离缝合处）最大差 = {np.abs(circ[1:-1]-lin[1:-1]).max():.2e}")
print("结论：差异只在首尾出现，符合「谱卷积=循环卷积、隐含周期延拓」的结构性判断。")

(a) 谱卷积与循环卷积最大差 = 2.78e-16
(b) 谱层复数权重形状 in×out×k = 32×32×8，可训练实参数 = 16384
(c) 非周期目标：边界点 |循环-零填充| = 4.92e-01
    内部（远离缝合处）最大差 = 1.23e-16
结论：差异只在首尾出现，符合「谱卷积=循环卷积、隐含周期延拓」的结构性判断。


## V6　文档一致性核对（B 组对 A 稿的静态检查）

核对两类可机检的事实：(1) A1 正文 6.1 表 4 的两列数字是否与其随稿数据文件 `A1组-6.4-网格结果.csv` 的列对应；
(2) A2 正文引用的数据文件是否真的存在于随稿 `数据/` 目录。

In [14]:
import os
ch6 = os.path.join(os.getcwd())     # 本 notebook 位于 工作/，直接使用相对路径
csv = os.path.join("数据", "A1组-6.4-网格结果.csv")
print("A1 数据文件存在:", os.path.exists(csv))
if os.path.exists(csv):
    import csv as _csv
    with open(csv, encoding="utf-8-sig") as f:
        rows_csv = list(_csv.DictReader(f))
    print("列名:", list(rows_csv[0].keys()))
    cols = list(rows_csv[0].keys())
    nm = [c for c in cols if "nodal" in c.lower() or "max" in c.lower()]
    l2 = [c for c in cols if "l2" in c.lower()]
    if nm and l2:
        for r in rows_csv[:6]:
            a = float(r[nm[0]]); b = float(r[l2[0]])
            print(f"  {nm[0]}={a:.6e}  {l2[0]}={b:.6e}  比值={b/a:.4f}")
        print("\n判据：若比值恒定 ≈0.447，说明正文把「相对L∞」写成了「相对L2」，两列其实是同一组数的不同归一化。")

refs = ["数据/A2组-6.4-逐问题指标.csv", "数据/A2组-6.4-逐种子指标.csv",
        "数据/A2组-6.4-复核记录.json", "数据/A2组-6.4-校验清单.json",
        "数据/A2组-6.4-数据基线.npz"]
print("\nA2 正文引用的数据文件：")
for r in refs:
    print(f"  {r}: {'存在' if os.path.exists(r) else '【缺失】'}")

# A1 正文 6.1 表 4：两列数字是否逐位相同
md_path = "A1-G12-正文.md"
print("\nA1 正文表 4 核对：", md_path, "存在" if os.path.exists(md_path) else "缺失")
if os.path.exists(md_path):
    txt = open(md_path, encoding="utf-8").read().splitlines()
    start = next((i for i, l in enumerate(txt) if "表 4" in l and "有限差分基线" in l), None)
    if start is not None:
        block = txt[start:start + 12]
        data = []
        for l in block:
            if not l.strip().startswith("|") or "---" in l:
                continue
            cells = [c.strip() for c in l.strip().strip("|").split("|")]
            if cells and cells[0].isdigit():
                data.append(cells)
        same = all(c[2] == c[3] for c in data)
        print(f"  表内数据行 {len(data)} 行；「相对L2」列与「最大误差」列逐位相同：{same}")
        for c in data:
            print(f"    N={c[0]}: 相对L2={c[2]}  最大误差={c[3]}")
        if same:
            print("  结论：两列写的是同一组数；而随稿 CSV 中 rel_l2/nodal_max 恒为 0.447，")
            print("        说明正文「相对L2」列实际填的是最大误差（nodal_max），列名与数值不符。")

A1 数据文件存在: True
列名: ['method', 'n', 'dof', 'rel_l2', 'rel_h1_semi', 'max_sample', 'boundary_max', 'solve_s', 'time_q25_s', 'time_q75_s', 'nodal_max', 'order_l2', 'order_h1']
  max_sample=3.290518e-04  rel_l2=1.471965e-04  比值=0.4473
  max_sample=8.225076e-05  rel_l2=3.678617e-05  比值=0.4472
  max_sample=2.056193e-05  rel_l2=9.195731e-06  比值=0.4472
  max_sample=5.140435e-06  rel_l2=2.298882e-06  比值=0.4472
  max_sample=1.285106e-06  rel_l2=5.747174e-07  比值=0.4472
  max_sample=4.931962e-04  rel_l2=3.603559e-04  比值=0.7307

判据：若比值恒定 ≈0.447，说明正文把「相对L∞」写成了「相对L2」，两列其实是同一组数的不同归一化。

A2 正文引用的数据文件：
  数据/A2组-6.4-逐问题指标.csv: 【缺失】
  数据/A2组-6.4-逐种子指标.csv: 【缺失】
  数据/A2组-6.4-复核记录.json: 【缺失】
  数据/A2组-6.4-校验清单.json: 【缺失】
  数据/A2组-6.4-数据基线.npz: 【缺失】

A1 正文表 4 核对： A1-G12-正文.md 存在
  表内数据行 5 行；「相对L2」列与「最大误差」列逐位相同：True
    N=51: 相对L2=$3.291\times10^{-4}$  最大误差=$3.291\times10^{-4}$
    N=101: 相对L2=$8.225\times10^{-5}$  最大误差=$8.225\times10^{-5}$
    N=201: 相对L2=$2.056\times10^{-5}$  最大误差=$2.056\times10^{-5}$
    N=

## 汇总

V1–V5 是 B 组自己的对照计算，V6 是静态一致性核对。测试结论与对应问题编号见
`B2组-验证报告.md` 的「测试证据」表；问题状态见 `B2组-修正清单.md`。